# 9주차 학생 워크북: 이미지 시각화 · 소량 이미지 분류

원, 사각형, 삼각형 PNG 각 20장(총 60장)을 시각화하고 HOG+SVM 및 Teachable Machine을 비교합니다. 클래스당 train 15장, test 5장입니다. 셀을 위에서부터 실행하세요.

In [ ]:
from pathlib import Path
import io, zipfile, requests
DATA_DIR=Path("/content/week09_shapes")
DATA_DIR.mkdir(parents=True,exist_ok=True)
url="https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/Week09/data/shapes60.zip"
res=requests.get(url,timeout=60)
res.raise_for_status()
with zipfile.ZipFile(io.BytesIO(res.content)) as z: z.extractall(DATA_DIR)
print("이미지:",len(list(DATA_DIR.rglob("*.png"))))
assert len(list(DATA_DIR.rglob("*.png")))==60

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
meta=pd.read_csv(DATA_DIR/"labels.csv")
display(meta.groupby(["split","label"]).size().unstack(fill_value=0))
fig,axes=plt.subplots(3,4,figsize=(9,7))
for i,label in enumerate(["circle","square","triangle"]):
    subset=meta[(meta.label==label)&(meta.split=="train")].head(4)
    for j,p in enumerate(subset.path):
        axes[i,j].imshow(Image.open(DATA_DIR/p))
        axes[i,j].set_title(label)
        axes[i,j].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
pixels=np.array(Image.open(DATA_DIR/meta.iloc[0]["path"]).convert("RGB"))
print("RGB 배열:",pixels.shape,"가운데 픽셀:",pixels[32,32].tolist())
fig,ax=plt.subplots(1,3,figsize=(12,3))
ax[0].imshow(pixels);ax[0].set_title("Original");ax[0].axis("off")
ax[1].imshow(pixels[:,:,0],cmap="Reds",vmin=0,vmax=255)
ax[1].set_title("Red channel");ax[1].axis("off")
for i,color in enumerate(["red","green","blue"]):
    ax[2].hist(pixels[:,:,i].ravel(),bins=20,range=(0,255),color=color,alpha=.4,label=color)
ax[2].set_title("RGB histogram");ax[2].legend()
plt.tight_layout();plt.show()

In [ ]:
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,classification_report,ConfusionMatrixDisplay
def features(path):
    gray=np.array(Image.open(DATA_DIR/path).convert("L"))
    return hog(gray,orientations=9,pixels_per_cell=(8,8),cells_per_block=(2,2),feature_vector=True)
X=np.stack([features(p) for p in meta.path])
y=meta.label.to_numpy()
train=(meta.split=="train").to_numpy()
X_train,X_test=X[train],X[~train]
y_train,y_test=y[train],y[~train]
print("학습:",len(y_train),"평가:",len(y_test))
assert len(y_train)==45 and len(y_test)==15

In [ ]:
model=SVC(kernel="linear",C=1.0,random_state=42)
model.fit(X_train,y_train)
pred=model.predict(X_test)
print("Colab 정확도:",round(accuracy_score(y_test,pred),3))
print(classification_report(y_test,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,labels=["circle","square","triangle"],cmap="Blues")
plt.show()
wrong=np.flatnonzero(pred!=y_test)
print("오분류 수:",len(wrong))
test_meta=meta[meta.split=="test"].reset_index(drop=True)
comparison=test_meta[["path","label"]].copy()
comparison["colab_prediction"]=pred
comparison["tm_prediction"]=""
display(comparison)
comparison.to_csv("Week09_Comparison.csv",index=False,encoding="utf-8-sig")

## 실습 B — Teachable Machine

1. [Teachable Machine](https://teachablemachine.withgoogle.com/)에서 Get Started → Image Project → Standard image model 선택.
2. [데이터 ZIP](https://github.com/itlects/2026-2_ai-data/raw/refs/heads/main/Week09/data/shapes60.zip)을 PC에 다운로드/압축 해제.
3. 클래스 circle / square / triangle 3개를 생성하고, 각 train 폴더의 PNG 15장씩 총 45장 업로드.
4. Train Model 실행.
5. test 폴더의 15장(클래스당 5장)을 Preview에 하나씩 입력. 가장 높은 예측 클래스를 기록.
6. Week09_Comparison.csv의 tm_prediction 열에 기록하고 정답 수/15로 정확도 계산.

**중요:** test 이미지를 Teachable Machine 훈련에 절대 넣지 마세요. 동일한 15장으로 두 방식을 비교합니다.

### 결과 보고 질문
- RGB 64×64×3에서 총 숫자 개수는?
- 원본, R 채널, RGB 히스토그램은 각각 무엇을 표현하는가?
- HOG 특징과 SVM은 각각 어떤 역할을 하는가?
- Colab과 Teachable Machine의 정확도는 각각 몇 %인가?
- 합성 도형에서 높은 정확도가 실제 사물 사진 분류를 보장하지 못하는 이유는?

**제출:** 이미지 시각화·히스토그램·혼동행렬 캡처, Teachable Machine 훈련·평가 화면 캡처, 평가 15장 비교표, 해석 3~5문장.

## 강의자 정답 참고

64×64×3 = 12,288개, 학습 45장/평가 15장. 합성 이미지에서의 정확도는 실제 사진에 일반화할 수 없습니다. HOG는 윤곽 방향 특징을 추출하며 SVM은 그 특징으로 클래스를 분류합니다.